## LG Aimers 9기 — F 예측 고정 라우팅 OOF 검증 (Colab)

07에서 저장한 행별 OOF를 다시 학습하지 않고 재조합한다. R 행에는 현재 CatBoost를 유지하고 F 행에만 사전 고정한 공유 CatBoost 정책을 사용한 뒤, 기존 LightGBM과 65:35 logit 평균을 다시 계산한다. 주 후보·대안·안전 조건은 아래에서 점수를 읽기 전에 고정하며 이 노트북은 제출 ZIP을 만들지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/LG Aimers 9th"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "experiments/07b_oof_routing.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "LG_aimers_9th 폴더 또는 experiments/07b_oof_routing.py를 찾지 못했습니다."
    )

required = [
    PROJECT_ROOT / "src/features.py",
    PROJECT_ROOT / "src/evaluation.py",
    PROJECT_ROOT / "src/metrics.py",
    PROJECT_ROOT / "experiments/07b_oof_routing.py",
    PROJECT_ROOT / "outputs/f_shift_cv_v1/latest_run.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0], "| CPU threads:", os.cpu_count())
print("pandas:", pd.__version__, "| NumPy:", np.__version__)

### 1. 07 OOF 입력과 기준 결과 확인

07의 자동 선택이 `current_all_nogt`였는지 확인하고, 최신 run의 `oof_predictions.csv.gz`를 입력으로 사용한다. 07b 스크립트는 필요한 확률 열의 범위, 746,504개 `row_id`의 유일성, 시즌과 경기 유형을 다시 검사한다.

In [ ]:
F_SHIFT_OUTPUT_ROOT = PROJECT_ROOT / "outputs/f_shift_cv_v1"
with (F_SHIFT_OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    f_shift_latest = json.load(handle)
recorded_source = Path(f_shift_latest["run_dir"])
SOURCE_RUN_DIR = next(
    (p for p in [recorded_source, F_SHIFT_OUTPUT_ROOT / recorded_source.name] if p.exists()),
    None,
)
if SOURCE_RUN_DIR is None:
    raise FileNotFoundError("07 최신 run 폴더를 찾지 못했습니다.")

SOURCE_OOF = SOURCE_RUN_DIR / "oof_predictions.csv.gz"
SOURCE_SUMMARY = SOURCE_RUN_DIR / "summary.json"
missing_source = [str(path) for path in [SOURCE_OOF, SOURCE_SUMMARY] if not path.exists()]
if missing_source:
    raise FileNotFoundError(f"07 결과 파일이 없습니다: {missing_source}")
with SOURCE_SUMMARY.open(encoding="utf-8") as handle:
    source_summary = json.load(handle)
input_check = pd.DataFrame([
    ["07 signature", source_summary["signature"]],
    ["07 selected policy", source_summary["selected_policy"]],
    ["07 base recent Brier", source_summary["base_recent_brier"]],
    ["07 base recent BSS", source_summary["base_recent_bss_raw"]],
    ["OOF size (MB)", round(SOURCE_OOF.stat().st_size / 1e6, 1)],
] , columns=["check", "value"])
display(input_check)
print("SOURCE_RUN_DIR:", SOURCE_RUN_DIR)

### 2. 라우팅 후보와 선택 우선순위 고정

같은 OOF에서 다시 많은 조합을 검색하지 않는다. 07 보고서에서 고정한 주 후보와 대안만 검증한다. `r_only F`는 최근 pooled가 높더라도 2022와 2024가 불안정했기 때문에 자동 선택에서 제외한다.

In [ ]:
route_plan = pd.DataFrame([
    ["current_r__current_f", "base", "current_all_nogt", False],
    ["current_r__latest_f1_f", "primary", "shared_latest_f1", True],
    ["current_r__fast_decay_f", "backup", "shared_f_fast_decay", True],
    ["current_r__r_only_f", "diagnostic_unstable", "r_only", False],
] , columns=["route", "role", "F CatBoost policy", "selection_eligible"])
route_plan.insert(2, "R CatBoost policy", "current_all_nogt")
display(route_plan)

주 후보의 고정 통과 조건은 다음과 같다.

- R 행의 CatBoost·최종 앙상블 예측이 기준과 수치적으로 완전히 동일
- 2023+2024 pooled 전체와 F Brier가 기준보다 개선
- 2022·2023·2024 각 fold에서 전체와 F Brier가 모두 기준보다 악화되지 않음
- 주 후보가 실패할 때만 같은 조건의 backup을 사용하고, 둘 다 실패하면 기준 유지

혼합비는 65:35 logit으로 고정하며 보정이나 추가 grid를 수행하지 않는다.

### 3. 07b OOF 라우팅 실행

모델 학습이 없으므로 CPU 런타임이면 충분하다. 전체 OOF CSV 읽기와 압축 저장에만 시간이 걸린다.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs/f_routing_v1"
command = [
    sys.executable,
    str(PROJECT_ROOT / "experiments/07b_oof_routing.py"),
    "--f-shift-output-root", str(F_SHIFT_OUTPUT_ROOT),
    "--source-run-dir", str(SOURCE_RUN_DIR),
    "--output-dir", str(OUTPUT_ROOT),
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

### 4. 결과와 입력 재현성 확인

In [ ]:
with (OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    latest = json.load(handle)
recorded_run = Path(latest["run_dir"])
RUN_DIR = next(
    (p for p in [recorded_run, OUTPUT_ROOT / recorded_run.name] if p.exists()),
    None,
)
if RUN_DIR is None:
    raise FileNotFoundError("07b 결과 run 폴더를 찾지 못했습니다.")
summary = latest["summary"]
source_audit = summary["source_audit"]
integrity = summary["integrity_audit"]
checks = pd.DataFrame([
    ["source rows", source_audit["rows"]],
    ["source signature", source_audit["source_signature"]],
    ["source selected policy", source_audit["source_selected_policy"]],
    ["base reconstruction max diff", integrity["base_blend_reconstruction_max_abs_difference"]],
    ["routing audit", integrity["status"]],
    ["selected route", summary["selected_route"]],
    ["selection status", summary["selection_status"]],
] , columns=["check", "value"])
display(checks)
print("RUN_DIR:", RUN_DIR)

### 5. 고정 라우팅 핵심 결과

Brier와 변화량은 낮을수록 좋다. 선택은 표의 최저 점수가 아니라 `primary → backup → base`의 사전 우선순위와 고정 gate로 결정된다.

In [ ]:
route_summary = pd.read_csv(RUN_DIR / "route_summary.csv")
route_summary["delta_recent_brier_x1e6"] = (
    1e6 * route_summary["delta_recent_brier_vs_base"]
)
route_summary["delta_recent_f_brier_x1e6"] = (
    1e6 * route_summary["delta_recent_f_brier_vs_base"]
)
core_columns = [
    "route", "selection_role", "recent_brier", "recent_bss_raw",
    "recent_mean_gap", "delta_recent_brier_x1e6",
    "delta_recent_f_brier_x1e6", "worst_fold_delta",
    "r_predictions_exact", "passes_fixed_gate",
]
display(route_summary.set_index("selection_role").loc[
    ["primary", "backup", "base", "diagnostic_unstable"]
].reset_index()[core_columns])

### 6. 2022·2023·2024 전체와 F 안정성

주 후보는 선택에 사용한 최근 pooled뿐 아니라 2022에서도 같은 방향이어야 한다. R은 기준 예측을 복사하므로 변화량이 정확히 0이어야 한다.

In [ ]:
fold_columns = [
    "route",
    *[f"delta_brier_{year}_vs_base" for year in [2022, 2023, 2024]],
    *[f"delta_brier_r_{year}_vs_base" for year in [2022, 2023, 2024]],
    *[f"delta_brier_f_{year}_vs_base" for year in [2022, 2023, 2024]],
]
fold_deltas = route_summary[fold_columns].set_index("route") * 1e6
display(fold_deltas)

plot_columns = [
    "delta_brier_2022_vs_base", "delta_brier_2023_vs_base",
    "delta_brier_2024_vs_base",
]
ax = (route_summary.set_index("route")[plot_columns] * 1e6).plot(
    kind="bar", figsize=(12, 5)
)
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("Overall Brier delta vs base × 1e6")
ax.set_title("Fixed route fold stability (lower is better)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### 7. 행 단위 라우팅 무결성

각 후보는 R에서 현재 CatBoost·현재 앙상블과 완전히 같고, F에서 지정 정책의 CatBoost·앙상블과 완전히 같아야 한다. 라우팅은 현재 한 행의 `game_type`만 이용한다.

In [ ]:
audit_rows = []
for route, values in integrity["route_audits"].items():
    audit_rows.append({"route": route, **values})
audit_table = pd.DataFrame(audit_rows)
audit_columns = [
    "route", "r_rows", "f_rows",
    "r_catboost_vs_current_max_abs_difference",
    "f_catboost_vs_policy_max_abs_difference",
    "r_blend_vs_current_max_abs_difference",
    "f_blend_vs_policy_max_abs_difference", "changed_r_rows",
]
display(audit_table[audit_columns])
assert integrity["routing_uses_only_current_row_game_type"]
assert audit_table["changed_r_rows"].eq(0).all()

### 8. Paired loss와 예측 변화

Paired loss는 같은 행에서 후보 제곱오차 minus 기준 제곱오차다. 음수가 개선이다. `naive_standard_error`는 경기 내 상관을 무시하므로 유의확률이 아니라 변화 규모 확인용이다.

In [ ]:
paired = pd.read_csv(RUN_DIR / "paired_loss_deltas.csv")
paired_focus = paired[
    paired["partition"].eq("recent_pooled")
    & paired["scope"].isin(["all", "R", "F"])
].sort_values(["scope", "mean_loss_delta_vs_base"])
display(paired_focus.reset_index(drop=True))

correlations = pd.read_csv(RUN_DIR / "prediction_correlations.csv")
correlation_focus = correlations[
    correlations["partition"].eq("recent_pooled")
    & correlations["scope"].isin(["all", "F"])
].sort_values(["scope", "mean_abs_prediction_delta_vs_base"], ascending=[True, False])
display(correlation_focus.reset_index(drop=True))

### 9. 자동 판정과 다음 단계

주 후보가 gate를 통과하면 다음 최종 학습에서는 현재 기본 CatBoost와 F 라우팅용 CatBoost를 각각 학습한다. 평가 스크립트는 각 행의 `game_type`으로 CatBoost 확률을 선택한 뒤 고정 LightGBM과 logit 평균한다. 07b 자체는 제출물이 아니다.

In [ ]:
decision = {
    "selected_route": summary["selected_route"],
    "selection_status": summary["selection_status"],
    "selected_passes_fixed_gate": summary["selected_passes_fixed_gate"],
    "selected_recent_brier": summary["selected_recent_brier"],
    "selected_recent_bss_raw": summary["selected_recent_bss_raw"],
    "delta_recent_brier_vs_base": summary["selected_delta_recent_brier_vs_base"],
    "next_step": summary["next_step"],
}
display(pd.DataFrame(decision.items(), columns=["item", "value"]))
if summary["selection_status"] == "primary_passed":
    print("판정: 주 라우팅 통과. 결과 검토 후 최종 학습·제출 노트북으로 이동합니다.")
elif summary["selection_status"] == "primary_failed_backup_passed":
    print("판정: 주 후보 실패, 고정 backup 통과. 결과 검토 후 backup을 재현합니다.")
else:
    print("판정: 라우팅 기각. 현재 submit_02를 유지합니다.")

### 10. 검토용 결과 ZIP 생성

In [ ]:
review_names = [
    "summary.json",
    "configuration.json",
    "route_manifest.csv",
    "integrity_audit.json",
    "route_summary.csv",
    "route_metrics.csv",
    "paired_loss_deltas.csv",
    "prediction_correlations.csv",
]
REVIEW_ZIP = OUTPUT_ROOT / "f_routing_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        path = RUN_DIR / name
        if not path.exists():
            raise FileNotFoundError(f"검토 파일이 없습니다: {path}")
        archive.write(path, arcname=name)

print("검토 ZIP:", REVIEW_ZIP)
print("크기 (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 2))
print("전달 파일: 실행 결과가 저장된 07b_oof_routing.ipynb + f_routing_review.zip")